<div align="right"><sub>Notebook 最終更新: 2026-04-23 09:15</sub></div>
<h1><strong>06. LLMエージェントの可視化</strong></h1>

前回（05回）で作成した「Writer（執筆者）」と「Editor（編集長）」による協調動作ループ（Executor-Critic パターン）は強力ですが、途中経過がログの文字としてダラダラ出力されるだけでは直感的に分かりづらいという課題がありました。

この回は、エージェントが内部でどのような「壁打ち（やり取り）」をしているかを、**Gradio** を使ってブラウザ上でスマートに確認できる UI を構築します。

In [ ]:
!pip install -q -U transformers accelerate bitsandbytes sentence-transformers faiss-cpu peft datasets gradio

import os
import sys
from google.colab import drive

DRIVE_MOUNT_POINT = '/content/drive'
drive.mount(DRIVE_MOUNT_POINT, force_remount=False)

PERSIST_ROOT = os.path.join(DRIVE_MOUNT_POINT, 'MyDrive', 'AIAgent')
PERSIST_INDEX_DIR = os.path.join(PERSIST_ROOT, 'data', 'index')
os.makedirs(PERSIST_INDEX_DIR, exist_ok=True)

REPO_ROOT = '/content/llm_lab'
if not os.path.exists(REPO_ROOT):
    !git clone -b ai_agent https://github.com/akio-kobayashi/llm_lab.git {REPO_ROOT}
else:
    !cd {REPO_ROOT} && git pull

os.chdir(REPO_ROOT)
src_path = os.path.abspath('src')
if src_path not in sys.path:
    sys.path.append(src_path)

print('現在の作業ディレクトリ:', os.getcwd())
from src.common import load_llm, generate_text, AGENT_MODEL_ID
from src.agent_core import LLMExecutorCriticAgent, RoleConfig
from src.ui import create_agent_ui

model, tokenizer = load_llm(model_id=AGENT_MODEL_ID)
print('準備完了')


In [ ]:
def llm_chat(system_prompt: str, user_prompt: str, max_tokens: int = 768, temp: float = 0.5):
    return generate_text(model, tokenizer, user_prompt, max_new_tokens=max_tokens, temperature=temp, system_prompt=system_prompt)

# 前回と同じ Writer と Editor の設定を準備します
writer_prompt = """
あなたはプロのライターです。ユーザーからのテーマについて、まずは標準的な解説記事を書いてください。
記事は必ず「解説」と「具体的な活用シーン」の両方を含めてください。
解説では、仕組み・特徴・社会への影響などを説明し、その後に読者がイメージできる具体的なシーンを示してください。
Editor（編集長）から修正指示が来た場合は、そのフィードバックを全面的に取り入れ、前回の文章から明確に改善された文章を書き直してください。
同じ内容の再提出は禁止です。
"""
editor_prompt = """
あなたは非常に厳しい編集長です。
提出された文章を読み、以下の3つの基準が【すべて】満たされているか評価してください。
1つ目は、テーマについての解説（仕組み・特徴・影響など）が明確に含まれていることです。単なる印象や描写ではなく、内容が理解できる説明になっているかを確認してください。
2つ目は、読者が日常生活でイメージしやすい具体的な活用シーンが含まれていることです。ただし、「通勤」などの単語だけでは不十分であり、時間・場所・行動が伴った情景として描写されているかを確認してください。
3つ目は、全体として読者が体験してみたいと感じるような、ワクワクする感情豊かなトーンになっていることです。ただし、「希望」「可能性」などの抽象的な表現だけに依存していないかも確認してください。
もし、1つでも満たしていない場合は、不足している点を具体的に指摘して書き直しを要求してください。
【重要】絶対に自分で文章を書き直さず、Writerへの修正指示だけを簡潔に出力してください。
すべて満たされていると判断した場合のみ、「誤りなし」と出力してください。
"""

writer = RoleConfig(name="Writer", system_prompt=writer_prompt)
editor = RoleConfig(name="Editor", system_prompt=editor_prompt)

agent = LLMExecutorCriticAgent(llm_chat, role_configs=[writer, editor])
print('エージェントの準備が完了しました。')

## **1. エージェントUIの起動**
以下のセルを実行して，UIを立ち上げます。前回コンソールに出力されていた「初稿 → 編集長のダメ出し → 第2稿」というステップが、サイドバーに綺麗に格納されていることを確認してください。

テスト用クエリ例: `「完全自動運転タクシー」が普及した社会について、300字程度で解説記事を書いてください。`

In [ ]:
def run_agent_for_ui(query):
    final_answer, full_log, steps = agent.run_pipeline(query, max_iterations=2)
    
    # ログを Markdown の引用形式に整形してアコーディオンに表示しやすくする
    formatted_log = ""
    for s in steps:
        formatted_log += f"### {s.role}\n> {s.observation.replace('\n', '\n> ')}\n\n"
    return final_answer, formatted_log

# Public URL (share=True) を発行してブラウザで確認します
ui = create_agent_ui(run_agent_for_ui)
ui.launch(share=True, debug=True)

## **2. 別の役割のエージェントを作成する**

「書く人」と「直す人」の役割を自由に変えることで、全く異なる品質向上を目指すことができます。
例えば、**「熱血営業マン（Writer）」** が書いた企画書を、**「超・論理的なリスク管理者（Critic）」** が添削するループを作ってみましょう。

In [ ]:
salesman_prompt = """
あなたは熱意ある営業担当者です。
商品の魅力を最大限に伝えつつ、顧客が安心して導入を判断できるように、利点とともに現実的な注意点も含めた「提案文」を作成してください。

リスク管理者から指摘を受けた場合は、魅力は維持したまま、リスクとその対策を明確に追加して書き直してください。
文章は500字以内で作成してください。
"""

risk_manager_prompt = """
あなたは冷徹で理詰めのリスク管理担当者です。
提出された提案文を読み、以下の2点が満たされているか評価してください：
	1.	現実的に想定されるリスクやデメリットが、少なくとも1つ「具体例（例：誤動作、故障、コストなど）」として記載されているか
	2.	そのリスクに対する具体的な対策やサポート体制が、少なくとも1つ明記されているか
上記2点が満たされていない場合のみ、不足点を指摘して修正を求めてください。
【重要】自分で文章を書き直さず、修正指示のみを出力してください。
【重要】上記2点が1つでも満たされていない場合は、「誤りなし」と出力してはいけません。
【重要】上記2点が両方とも満たされている場合は、必ず「誤りなし」と出力し、それ以上の指摘をしてはなりません。
"""

custom_roles = [
    RoleConfig(name="Salesman", system_prompt=salesman_prompt),
    RoleConfig(name="RiskManager", system_prompt=risk_manager_prompt)
]
custom_agent = LLMExecutorCriticAgent(llm_chat, role_configs=custom_roles)

def run_custom_agent_ui(query):
    final_answer, full_log, steps = custom_agent.run_pipeline(query, max_iterations=2)
    formatted_log = ""
    for s in steps:
        formatted_log += f"### {s.role}\n> {s.observation.replace('\n', '\n> ')}\n\n"
    return final_answer, formatted_log

ui_custom = create_agent_ui(run_custom_agent_ui)
ui_custom.launch(share=True, debug=True)

# テスト用クエリ例: 「AIが全自動で家事をしてくれるホームロボットについて、導入を検討している顧客向けに、商品の魅力だけでなく、現実的なリスクやデメリット、およびその対策やサポート体制も含めた提案文（営業文）を作成してください。」

## **まとめ**
- UIを通じてAIエージェントの処理過程を可視化することで、「AIが裏側でどうやって賢く修正を繰り返しているか」をユーザーに分かりやすく伝えることができます。
- これは、単に便利だからだけでなく、「AIシステムをデバッグ・改善する（どの役割のプロンプトを直すべきか特定する）」上で非常に重要なプロセスです。